In [6]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path.cwd().parents[0]
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Raw files:", list(RAW_DIR.glob("*")))


Project root: /Users/sohithmalyala/Desktop/Projects/demand-forecasting-platform
Raw files: [PosixPath('/Users/sohithmalyala/Desktop/Projects/demand-forecasting-platform/data/raw/.DS_Store'), PosixPath('/Users/sohithmalyala/Desktop/Projects/demand-forecasting-platform/data/raw/calendar.csv'), PosixPath('/Users/sohithmalyala/Desktop/Projects/demand-forecasting-platform/data/raw/sell_prices.csv'), PosixPath('/Users/sohithmalyala/Desktop/Projects/demand-forecasting-platform/data/raw/sales_train_validation.csv')]


In [2]:
sales = pd.read_csv(RAW_DIR / "sales_train_validation.csv")
calendar = pd.read_csv(RAW_DIR / "calendar.csv")
prices = pd.read_csv(RAW_DIR / "sell_prices.csv")

print("Sales shape:", sales.shape)
print("Calendar shape:", calendar.shape)
print("Prices shape:", prices.shape)


Sales shape: (30490, 1919)
Calendar shape: (1969, 14)
Prices shape: (6841121, 4)


In [3]:
id_cols = ["id","item_id","dept_id","cat_id","store_id","state_id"]
d_cols = [c for c in sales.columns if c.startswith("d_")]

sales_long = sales.melt(
    id_vars=id_cols,
    value_vars=d_cols,
    var_name="d",
    value_name="y"
)

print("sales_long:", sales_long.shape)
sales_long.head()


sales_long: (58327370, 8)


,id,item_id,dept_id,cat_id,store_id,state_id,d,y
0,HOBBIES_1_001_CA_1_validation,HOBBIES_1_001,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
1,HOBBIES_1_002_CA_1_validation,HOBBIES_1_002,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
2,HOBBIES_1_003_CA_1_validation,HOBBIES_1_003,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
3,HOBBIES_1_004_CA_1_validation,HOBBIES_1_004,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0
4,HOBBIES_1_005_CA_1_validation,HOBBIES_1_005,HOBBIES_1,HOBBIES,CA_1,CA,d_1,0


In [4]:
calendar["date"] = pd.to_datetime(calendar["date"])

sales_long = sales_long.merge(calendar, on="d", how="left")
sales_long = sales_long.merge(prices, on=["store_id","item_id","wm_yr_wk"], how="left")

print("after joins:", sales_long.shape)
sales_long[["id","date","y","sell_price","event_name_1","snap_CA"]].head()


after joins: (58327370, 22)


,id,date,y,sell_price,event_name_1,snap_CA
0,HOBBIES_1_001_CA_1_validation,2011-01-29,0,NaN,NaN,0
1,HOBBIES_1_002_CA_1_validation,2011-01-29,0,NaN,NaN,0
2,HOBBIES_1_003_CA_1_validation,2011-01-29,0,NaN,NaN,0
3,HOBBIES_1_004_CA_1_validation,2011-01-29,0,NaN,NaN,0
4,HOBBIES_1_005_CA_1_validation,2011-01-29,0,NaN,NaN,0


In [9]:
out_path = PROCESSED_DIR / "sales_long.parquet"
sales_long["y"] = sales_long["y"].astype("int32")
sales_long.to_parquet(out_path, index=False)

print("✅ saved:", out_path)


✅ saved: /Users/sohithmalyala/Desktop/Projects/demand-forecasting-platform/data/processed/sales_long.parquet


In [10]:
import os
os.path.exists(PROCESSED_DIR / "sales_long.parquet")


True

In [11]:
(PROCESSED_DIR / "sales_long.parquet").stat().st_size / 1e9


0.285709168

In [12]:
print("sales_long shape:", sales_long.shape)
print("Missing date %:", sales_long["date"].isna().mean() * 100)
print("Negative y count:", (sales_long["y"] < 0).sum())
print("Duplicate (id,date):", sales_long.duplicated(["id","date"]).sum())
print("Missing sell_price %:", sales_long["sell_price"].isna().mean() * 100)


sales_long shape: (58327370, 22)
Missing date %: 0.0
Negative y count: 0
Duplicate (id,date): 0
Missing sell_price %: 21.08686367994991
